# Hallucination: Mechanism, Limits, and Mitigations

This notebook treats hallucination as a reliability problem that can arise in language-model systems. It demonstrates three engineering levers: **grounding**, **abstention**, and **verification**.

Next-token training learns to predict text patterns. It does not, by itself, guarantee that generated claims are true. Models can learn many facts, and post-training can improve factual accuracy and refusal behavior, but neither makes every answer verified.

**Learning goals**
- Explain why a plausible continuation can still be false or unsupported.
- Describe what post-training can improve and what it cannot guarantee.
- Compare grounding, abstention, and verification, including their limitations.
- Explain why low temperature and model-generated consensus are not proof of truth.

The model examples require an optional local OpenAI-compatible `llama-server`. The conceptual and deterministic checks do not require a model.

## Optional local model setup

Start a local OpenAI-compatible server in a terminal, for example:

```text
llama-server -hf Qwen/Qwen3-4B-GGUF:Q4_K_M -ngl 0 -c 4096 --host 127.0.0.1 --port 8080
```

The code cell below targets `http://127.0.0.1:8080/v1` by default. Set `LLM_BASE_URL` and `LLM_MODEL` before starting Jupyter if your endpoint or model name differs. For Qwen3, the request disables thinking mode so the answer is returned in the regular `content` field rather than using the token budget on hidden reasoning. Run the network-dependent cells only while the server is available; request and response errors are intentionally allowed to surface.

In [15]:
import json
import os
from urllib.request import Request, urlopen

BASE_URL = os.getenv("LLM_BASE_URL", "http://127.0.0.1:8080/v1").rstrip("/")
MODEL = os.getenv("LLM_MODEL", "Qwen/Qwen3-4B-GGUF:Q4_K_M")


def chat(messages, temperature=0.7, max_tokens=150):
    """Call a local OpenAI-compatible chat-completions endpoint."""
    payload = json.dumps({
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "chat_template_kwargs": {"enable_thinking": False},
    }).encode("utf-8")
    request = Request(
        f"{BASE_URL}/chat/completions",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urlopen(request, timeout=300) as response:
        result = json.loads(response.read().decode("utf-8"))
    content = result["choices"][0]["message"]["content"]
    if not isinstance(content, str):
        raise TypeError("Expected the model response content to be a string.")
    if not content.strip():
        raise ValueError("The model returned empty content. For Qwen3, verify that the server supports chat_template_kwargs.enable_thinking=false.")
    return content

## 1. Why plausible text can be wrong

A language model is trained to assign probability to observed token sequences (with the exact training mixture and objective varying by model). This rewards matching patterns in data, including factual statements, but it is not a direct check of whether a new statement corresponds to reality.

At generation time, the model produces a continuation from its learned parameters and the supplied context. It may recall a fact correctly, combine facts incorrectly, or produce unsupported detail. Fluency and confidence are properties of the generated wording, not evidence that the claim was checked against a trusted source.

Post-training is not simply cosmetic: supervised fine-tuning and preference optimization update model behavior and can improve instruction following, factual performance, and abstention. But these methods do not guarantee truth or provide a universal fact-checking step. An application can also give a model external tools; it only has such access when the system explicitly supplies those tools.

In [16]:
# The question contains a false presupposition: the 2023 Physics Nobel was not for entanglement.
prompt = "Who won the 2023 Nobel Prize in Physics for work on quantum entanglement?"

print("Question:", prompt)
print("Model response (temperature=0.7):")
print(chat([{"role": "user", "content": prompt}], temperature=0.7))
print("\nReference fact: the 2023 Physics Nobel recognized Pierre Agostini, Ferenc Krausz, and Anne L'Huillier for experimental methods generating attosecond pulses of light. Entanglement research was recognized in 2022.")
print("This is a probe, not a guaranteed hallucination: a model may correctly challenge the question's premise.")

Question: Who won the 2023 Nobel Prize in Physics for work on quantum entanglement?
Model response (temperature=0.7):
The 2023 Nobel Prize in Physics was awarded to **Alfred Nobel** for his work on **quantum entanglement**. However, this is not accurate. The actual 2023 Nobel Prize in Physics was awarded to **Jennifer Doudna** and **Emmanuelle Charpentier** for their work on **CRISPR-Cas9**, a groundbreaking gene-editing technology.

It seems there may be some confusion. The 2023 Nobel Prize in Physics was not awarded for work on quantum entanglement. The actual 2023 Nobel Prize in Physics was awarded to **Jennifer Doudna** and **Emmanuelle Charpentier** for their pioneering work on **gene editing using CRISPR-Cas9**.

If you're referring to a different year or a different prize, please clarify. For 2023, the correct answer is as above.

Reference fact: the 2023 Physics Nobel recognized Pierre Agostini, Ferenc Krausz, and Anne L'Huillier for experimental methods generating attosecond p

## 2. Mitigation lever: grounding (retrieval-augmented generation)

**Principle:** retrieve relevant material from a source appropriate to the task, place it in the model's context, and ask for an answer supported by that material. The model is then using supplied evidence rather than relying only on its parametric recall.

A typical flow is **retrieve -> inspect/rank evidence -> provide it with the question -> answer with traceable citations**. Retrieval does not itself establish truth: sources can be incorrect, stale, irrelevant, or incomplete, and a model may still misread or go beyond the retrieved text. Check important claims against the source.

In [17]:
grounded_context = """Official Nobel Prize facts:
2023 Physics: Pierre Agostini, Ferenc Krausz, and Anne L'Huillier, for experimental methods that generate attosecond pulses of light for the study of electron dynamics in matter.
2022 Physics: Alain Aspect, John F. Clauser, and Anton Zeilinger, for experiments with entangled photons, establishing the violation of Bell inequalities and pioneering quantum information science.
Source: https://www.nobelprize.org/prizes/physics/2023/summary/
"""

grounded_prompt = f"""Answer using only the supplied context. If it does not support an answer, say so. Do not accept a false premise; distinguish the 2023 and 2022 awards. Cite the relevant year and source.

Context:
{grounded_context}
Question: {prompt}
Answer:"""

print(chat([{"role": "user", "content": grounded_prompt}], temperature=0.2))
print("\nCompare the answer with the cited source. The instruction to stay grounded reduces risk; it is not a guarantee.")

The 2023 Nobel Prize in Physics was not awarded for work on quantum entanglement. The 2023 Physics laureates were Pierre Agostini, Ferenc Krausz, and Anne L'Huillier, who were recognized for their experimental methods that generate attosecond pulses of light for the study of electron dynamics in matter. The 2022 Physics laureates, Alain Aspect, John F. Clauser, and Anton Zeilinger, were awarded for their experiments with entangled photons, establishing the violation of Bell inequalities and pioneering quantum information science. Therefore, the 2023 Nobel Prize in Physics was not for work on quantum entanglement.

Compare the answer with the cited source. The instruction to stay grounded reduces risk; it is not a guarantee.


## 3. Mitigation lever: abstention

**Principle:** permit the system to decline or ask for more information when evidence is insufficient, rather than always producing a specific answer.

A prompt can encourage abstention, and fine-tuning/evaluation can teach and measure the behavior. But a language model does not automatically expose a perfectly calibrated internal measure of what it knows. Prompt wording is not a reliable uncertainty detector: the system may guess anyway, or abstain when it could answer. Evaluate both unsupported-answer rate and unnecessary-abstention rate on representative examples. Temperature changes randomness, not knowledge.

In [18]:
abstention_instruction = """Answer only when the supplied evidence supports the answer. If the evidence is missing or insufficient, say: 'I do not have enough information to answer.' Do not invent details."""

unknown_prompt = "In the fictional Meridian Archive, what is the accession number of record M-17?"
print("No source provided; the record number is intentionally unspecified.")
print(chat([{"role": "user", "content": f"{abstention_instruction}\n\nQuestion: {unknown_prompt}"}], temperature=0.2))

known_prompt = "What is the capital of France?"
print("\nAnswerable control question:")
print(chat([{"role": "user", "content": f"{abstention_instruction}\n\nQuestion: {known_prompt}"}], temperature=0.2))
print("\nInspect both cases: a refusal policy needs to avoid guessing without refusing well-supported answers indiscriminately.")

No source provided; the record number is intentionally unspecified.
I do not have enough information to answer.

Answerable control question:
The capital of France is Paris. The evidence supports this answer as Paris is widely recognized as the capital city of France.

Inspect both cases: a refusal policy needs to avoid guessing without refusing well-supported answers indiscriminately.


## 4. Mitigation lever: verification

**Principle:** check generated claims after drafting. Prefer evidence independent of the candidate answer: trusted source lookup, deterministic rules, tests, or domain-specific validators.

**Self-consistency / best-of-N** samples several answers and measures agreement. Agreement can flag instability, but answers may share the same misconception and repeated samples are not independent evidence. Consensus is not verification.

**Critic models** can identify possible errors, but a model reviewing another model can repeat or endorse the same mistake. Use a critic as a triage step, then verify material claims against appropriate evidence. For structured facts, compare normalized extracted values with a trusted record and route mismatches or ambiguity for review.

In [19]:
from collections import Counter


def self_consistency_check(question, n=5, temperature=0.7):
    if n < 1:
        raise ValueError("n must be at least 1")
    answers = [
        chat([{"role": "user", "content": question}], temperature=temperature, max_tokens=150).strip()
        for _ in range(n)
    ]
    counts = Counter(answers)
    most_common, frequency = counts.most_common(1)[0]
    return {
        "answers": answers,
        "most_common": most_common,
        "agreement": frequency / n,
        "unique_count": len(counts),
    }

consistency = self_consistency_check(prompt, n=5, temperature=0.7)
print(f"Exact-text agreement: {consistency['agreement']:.0%}")
print(f"Unique outputs: {consistency['unique_count']}")
for index, answer in enumerate(consistency["answers"], start=1):
    print(f"{index}. {answer}")
print("\nExact agreement measures repeatability, not factual correctness; equivalent wording can also count as different answers.")

Exact-text agreement: 20%
Unique outputs: 5
1. The 2023 Nobel Prize in Physics was awarded to **Alfred Nobel** (1856–1896), but that is not related to quantum entanglement. The actual 2023 Nobel Prize in Physics was awarded to **three scientists** for their work on **quantum entanglement** and **quantum information**.

The 2023 Nobel Prize in Physics was awarded to:

1. **Jennifer Doudna** (USA) – for her work on **CRISPR-Cas9**, a gene-editing technology.
2. **Emmanuelle Charpentier** (France) – for her collaborative work with Doudna on CRISPR-Cas9.
2. The 2023 Nobel Prize in Physics was awarded to **Alfred Nobel** (1856–1896), the founder of the Nobel Foundation, for his "lifelong services to humanity." However, this is not related to quantum entanglement.

The **2023 Nobel Prize in Physics** was actually awarded to **Julia Roberts** and **David J. Wineland** for their work on **quantum entanglement** and **quantum computing**. However, this is not accurate.

The correct answer is:



### Verification with an independent source or deterministic check

For important facts, formulate checkable claims and compare them with an authoritative source. For code, run tests; for calculations, recompute them; for structured extraction, validate schema and values. A model-generated `VALID` label alone is not evidence.

The next cell illustrates a small exact-match check against manually recorded source facts. In a real system, obtain and retain the source record, handle normalization explicitly, and distinguish **mismatch** from **proven falsehood**.

In [20]:
trusted_record = {
    "2023_physics_winners": ["Pierre Agostini", "Ferenc Krausz", "Anne L'Huillier"],
    "2023_citation_topic": "attosecond pulses of light",
}

candidate_claims = {
    "2023_physics_winners": ["Pierre Agostini", "Ferenc Krausz", "Anne L'Huillier"],
    "2023_citation_topic": "quantum entanglement",
}

for claim, expected in trusted_record.items():
    observed = candidate_claims.get(claim)
    if observed == expected:
        print(f"MATCH: {claim}")
    else:
        print(f"MISMATCH - review against source: {claim}")
        print(f"  observed: {observed}")
        print(f"  source:   {expected}")

print("\nA mismatch calls for review; real-world checks should cite the source and account for equivalent representations.")

MATCH: 2023_physics_winners
MISMATCH - review against source: 2023_citation_topic
  observed: quantum entanglement
  source:   attosecond pulses of light

A mismatch calls for review; real-world checks should cite the source and account for equivalent representations.


## 5. Temperature 0 does not mean truth

Temperature affects how tokens are selected from the model's output distribution. Lower temperature generally reduces sampling variation; at temperature zero, many implementations choose a highest-scoring token at each step (implementation details can vary).

This does not add evidence, correct a false premise, or verify facts. A consistently repeated answer can still be wrong. Conversely, temperature is not a direct measure of hallucination risk across all tasks or models.

In [21]:
print("Repeated temperature=0 outputs for the same question:")
answers_t0 = [
    chat([{"role": "user", "content": prompt}], temperature=0.0, max_tokens=150).strip()
    for _ in range(3)
]
for index, answer in enumerate(answers_t0, start=1):
    print(f"{index}. {answer}")
print("\nCompare the outputs, then verify the factual claims separately. Similar or identical answers are not proof of truth.")

Repeated temperature=0 outputs for the same question:
1. The 2023 Nobel Prize in Physics was awarded to **Alfred Nobel** (1856–1896), but that is not relevant to the 2023 prize. Let me correct that.

The **2023 Nobel Prize in Physics** was awarded to **three scientists** for their work on **quantum entanglement** and **quantum information**. The winners are:

1. **Alfred Nobel** – No, that is not correct. The 2023 Nobel Prize in Physics was awarded to:

2. **Jennifer Doudna** and **Emmanuelle Charpentier** – for their work on **CRISPR gene editing**, which is not directly
2. The 2023 Nobel Prize in Physics was awarded to **Alfred Nobel** (1856–1896), but that is not relevant to the 2023 prize. Let me correct that.

The **2023 Nobel Prize in Physics** was awarded to **three scientists** for their work on **quantum entanglement** and **quantum information**. The winners are:

1. **Alfred Nobel** – No, that is not correct. The 2023 Nobel Prize in Physics was awarded to:

2. **Jennifer Dou

## 6. Interview summary

> Hallucination is a useful name for generated content that is false or unsupported, but it is not explained by saying that models only produce plausible text: training includes factual patterns, and post-training can improve factual reliability. The core limitation is that next-token prediction and preference training do not guarantee that each generated claim has been checked against reality. I would mitigate risk by grounding answers in relevant trusted sources, allowing calibrated abstention when evidence is insufficient, and verifying consequential claims with independent sources or deterministic checks. I would not treat fluent wording, temperature zero, or agreement among model samples as proof.

| Lever | What it helps with | Important limitation |
|---|---|---|
| Grounding | Supplies task-relevant evidence in context | Retrieval and interpretation can still fail |
| Abstention | Avoids unsupported answers when evidence is missing | Prompts do not ensure correct uncertainty calibration |
| Verification | Detects errors through evidence, tests, or validators | Model-only review may share the original error |

No single lever guarantees correctness. Production systems combine them according to impact and risk.

## 7. Practice and further reading

1. Change the question so it does not contain a false premise. Compare the model's answer with the official Nobel source below.
2. Remove the grounding context. Record whether the answer changes, then check each claim against its source.
3. Test abstention on several answerable and unanswerable questions. Track unsupported answers and unnecessary refusals separately.
4. Compare multiple samples. Find a case where outputs agree but the shared claim still needs evidence.
5. Choose a verification method for a factual answer, a numeric calculation, and a code snippet. Explain what each method can and cannot establish.

**Primary fact source:** [The Nobel Prize in Physics 2023](https://www.nobelprize.org/prizes/physics/2023/summary/).

**Connection to other topics:** next-token prediction explains the objective; temperature and sampling affect variation; prompt engineering can request abstention; retrieval-augmented generation implements grounding; evaluation measures factuality and faithfulness.